In [24]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [25]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

In [26]:
# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

In [27]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}") # data
    print(f"Shape of y: {y.shape} {y.dtype}") # labels
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


In [28]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [29]:
# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [30]:
learning_rate = 1e-3 # step size for the optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [31]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad() # autograd accumulates gradients everytime loss.backward is called
        # zeroing means this batch is not affected by gradients from previous batches

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [32]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [11]:
epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 1.178490  [   64/60000]
loss: 1.168490  [ 6464/60000]
loss: 1.000666  [12864/60000]
loss: 1.122046  [19264/60000]
loss: 0.993485  [25664/60000]
loss: 1.037453  [32064/60000]
loss: 1.066845  [38464/60000]
loss: 1.008991  [44864/60000]
loss: 1.049196  [51264/60000]
loss: 0.980702  [57664/60000]
Test Error: 
 Accuracy: 65.6%, Avg loss: 0.994645 

Epoch 2
-------------------------------
loss: 1.057641  [   64/60000]
loss: 1.069261  [ 6464/60000]
loss: 0.886520  [12864/60000]
loss: 1.032080  [19264/60000]
loss: 0.902183  [25664/60000]
loss: 0.942409  [32064/60000]
loss: 0.988967  [38464/60000]
loss: 0.933187  [44864/60000]
loss: 0.967176  [51264/60000]
loss: 0.913561  [57664/60000]
Test Error: 
 Accuracy: 66.7%, Avg loss: 0.920922 

Epoch 3
-------------------------------
loss: 0.967885  [   64/60000]
loss: 0.999689  [ 6464/60000]
loss: 0.804006  [12864/60000]
loss: 0.967925  [19264/60000]
loss: 0.840829  [25664/60000]
loss: 0.872039  [32064/600

### Test space

In [34]:
# pool of square window of size=3, stride=2
m = nn.MaxPool2d(2)
# pool of non-square window
input = torch.randn(20, 16, 50, 32)
output = m(input)

In [52]:
input.shape

torch.Size([20, 4, 50, 100])

In [55]:
output.shape

torch.Size([20, 33, 50, 100])

In [54]:
# With square kernels and equal stride
m = nn.Conv2d(4, 33, kernel_size=3, stride=1, dilation=1, padding=1)
input = torch.randn(20, 4, 50, 100)
output = m(input)

In [58]:
x1 = torch.randn(20, 4, 50, 100)
x2 = torch.randn(20, 4, 50, 100)

torch.cat([x1,x2], dim=1).shape

torch.Size([20, 8, 50, 100])